# Describe Feature Services

Inventory ArcGIS Online hosted **feature services** into a data dictionary. For each service this records item metadata, every layer/table, geometry type, record count, spatial reference, and per-field **name / alias / type / length / nullability / domain** (coded values *or* range). Writes the log as TXT, CSV (flat, one row per field), or **multi-sheet XLSX** (`Services` / `Layers` / `Fields`).

**How to use:** edit the **CONFIG** cell, then *Run All*. Sign-in uses your active ArcGIS session via `GIS("home")` (ArcGIS Notebooks) or `GIS("pro")` (ArcGIS Pro) -- no credentials live in this notebook.

**Make it yours:** the describe functions are written **PropertyMap-OR-dict tolerant**, so the writers serialize cleanly and can be unit-tested with plain dicts (no portal needed). Adapted from an All Things Spatial workflow.

Runtime: arcgispro-py3 / ArcGIS Notebooks. Needs the ArcGIS API for Python + `openpyxl` for XLSX output (both ship with ArcGIS Pro).

## 1 - Config

In [ ]:
# ============================================================================
# CONFIG -- edit these, then Run All. Nothing else needs changing for a basic run.
# ============================================================================
SCOPE         = "entire"        # "entire" | "folders" | "single" | "search"
FOLDERS       = []              # when SCOPE="folders"; "(root)" = the root folder
ID_OR_URL     = ""             # when SCOPE="single": an item ID or a service URL
SEARCH_TEXT   = ""             # when SCOPE="search": title text (your items only)
OUTPUT_DIR    = "agol_inventory"   # local folder for the log file
OUTPUT_NAME   = ""             # blank = auto (agol_inventory_<UTC timestamp>)
OUTPUT_FORMAT = "XLSX"         # "XLSX" | "CSV" | "TXT"
COUNT_RECORDS = True            # one count query per layer; slow on huge services
MAX_ITEMS     = 500             # safety cap on services processed

## 2 - Describe core (PropertyMap-OR-dict tolerant)

In [ ]:
import csv
import datetime
import json
import os
from pathlib import Path


def _noop(*_a, **_k):
    pass


def _g(obj, key, default=None):
    """Get `key` from a dict OR an attribute-style object (arcgis PropertyMap)."""
    if obj is None:
        return default
    if isinstance(obj, dict):
        return obj.get(key, default)
    return getattr(obj, key, default)


def describe_field(f):
    """One field -> a flat dict, with domain flattened to a readable string."""
    domain = _g(f, "domain")
    dtype = dname = dvalues = ""
    if domain:
        dtype = _g(domain, "type", "") or ""
        dname = _g(domain, "name", "") or ""
        if "codedValue" in dtype or _g(domain, "codedValues"):
            pairs = _g(domain, "codedValues", []) or []
            dtype = "codedValue"
            dvalues = "; ".join("{}={}".format(_g(cv, "code"), _g(cv, "name")) for cv in pairs)
        elif "range" in dtype or _g(domain, "range"):
            rng = _g(domain, "range", []) or []
            dtype = "range"
            dvalues = "min={}, max={}".format(*(rng + ["", ""])[:2])
    return {
        "name": _g(f, "name", ""),
        "alias": _g(f, "alias", ""),
        "type": str(_g(f, "type", "")).replace("esriFieldType", ""),
        "length": _g(f, "length", ""),
        "nullable": _g(f, "nullable", ""),
        "domainType": dtype,
        "domainName": dname,
        "domainValues": dvalues,
    }


def describe_layer(layer, kind="layer", count_records=True, say=_noop):
    """One FeatureLayer/Table -> dict. `layer` needs `.properties` and (for counts)
    a `.query(return_count_only=True)` method."""
    p = _g(layer, "properties", layer)        # tolerate being handed props directly
    extent = _g(p, "extent")
    wkid = _g(_g(extent, "spatialReference"), "latestWkid") or \
        _g(_g(extent, "spatialReference"), "wkid")
    count = ""
    if count_records:
        try:
            count = layer.query(where="1=1", return_count_only=True)
        except Exception as e:  # noqa: BLE001
            say("    (count failed: {})".format(e))
            count = "n/a"
    fields = _g(p, "fields", []) or []
    return {
        "name": _g(p, "name", ""),
        "id": _g(p, "id", ""),
        "kind": kind,
        "geometryType": str(_g(p, "geometryType", "") or "").replace("esriGeometryType", "").replace("esriGeometry", ""),
        "recordCount": count,
        "wkid": wkid or "",
        "fieldCount": len(fields),
        "fields": [describe_field(f) for f in fields],
    }


def describe_item(item, count_records=True, say=_noop):
    """An arcgis Item (hosted feature service) -> dict incl. each layer + table."""
    say("Describing: {} ({})".format(_g(item, "title", "?"), _g(item, "id", "?")))
    layers = []
    for lyr in (getattr(item, "layers", None) or []):
        layers.append(describe_layer(lyr, "layer", count_records, say))
    for tbl in (getattr(item, "tables", None) or []):
        layers.append(describe_layer(tbl, "table", count_records, say))
    return {
        "title": _g(item, "title", ""),
        "id": _g(item, "id", ""),
        "owner": _g(item, "owner", ""),
        "type": _g(item, "type", ""),
        "url": _g(item, "url", ""),
        "created": _epoch(_g(item, "created")),
        "modified": _epoch(_g(item, "modified")),
        "access": _g(item, "access", ""),
        "tags": ", ".join(_g(item, "tags", []) or []),
        "snippet": _g(item, "snippet", "") or "",
        "layers": layers,
    }


def _epoch(ms):
    """AGOL stores ms-epoch; return an ISO date (or '')."""
    try:
        return datetime.datetime.fromtimestamp(int(ms) / 1000, datetime.timezone.utc).strftime("%Y-%m-%d")
    except Exception:  # noqa: BLE001
        return ""

## 3 - Gather services per scope (takes an already-connected `gis`)

In [ ]:
_FS_TYPES = ("Feature Service", "Feature Layer")


def _is_feature_service(item):
    return _g(item, "type", "") in _FS_TYPES


def gather_services(gis, scope, folders=None, item_id_or_url=None,
                    search_query=None, max_items=500):
    """Return a list of feature-service Items per scope:
    'entire' | 'folders' | 'single' | 'search'. A non-empty item_id_or_url always
    wins (direct selection of one service, owned or not)."""
    # Direct id / URL override -- highest priority.
    if item_id_or_url:
        token = item_id_or_url.strip()
        item = None
        if token.lower().startswith("http"):
            item = _item_from_url(gis, token)
        else:
            item = gis.content.get(token)
        return [item] if item else []

    me = gis.users.me
    if scope == "search":
        text = (search_query or "").replace('"', "").strip()
        q = 'title:"{}" owner:{}'.format(text, me.username) if text else "owner:{}".format(me.username)
        items = gis.content.search(q, item_type="Feature *", max_items=max_items)
        return [it for it in items if _is_feature_service(it)]

    if scope == "single":
        return []   # handled via item_id_or_url upstream

    # entire / folders -> walk owner items by folder
    if scope == "folders":
        targets = [None if f == "(root)" else f for f in (folders or [])]
    else:
        names = [f.get("title") if isinstance(f, dict) else getattr(f, "name", None)
                 for f in (me.folders or [])]
        # root is already the None entry; 2.4.x also lists it as "Root Folder"
        targets = [None] + [n for n in names if n != "Root Folder"]
    out, seen = [], set()
    for folder in targets:
        try:
            items = me.items(folder=folder, max_items=max_items)
        except Exception:
            items = []
        for it in items:
            if it.id not in seen and _is_feature_service(it):
                seen.add(it.id)
                out.append(it)
                if len(out) >= max_items:
                    return out
    return out


def _item_from_url(gis, url):
    """Best-effort: resolve a service URL to its portal Item."""
    try:
        from arcgis.features import FeatureLayerCollection
        flc = FeatureLayerCollection(url, gis)
        if getattr(flc, "properties", None) is not None:
            sid = _g(flc.properties, "serviceItemId")
            if sid:
                return gis.content.get(sid)
    except Exception:
        pass
    return None

## 4 - Writers (TXT / CSV / multi-sheet XLSX)

In [ ]:
def _field_rows(report):
    """Flatten report -> one row per field (for CSV / the Fields sheet)."""
    rows = []
    for svc in report["services"]:
        for lyr in svc["layers"]:
            for f in lyr["fields"]:
                rows.append({
                    "service_title": svc["title"], "service_id": svc["id"],
                    "layer_name": lyr["name"], "layer_kind": lyr["kind"],
                    "geometryType": lyr["geometryType"], "recordCount": lyr["recordCount"],
                    "field_name": f["name"], "alias": f["alias"], "type": f["type"],
                    "length": f["length"], "nullable": f["nullable"],
                    "domainType": f["domainType"], "domainName": f["domainName"],
                    "domainValues": f["domainValues"],
                })
    return rows


_FIELD_COLS = ["service_title", "service_id", "layer_name", "layer_kind",
               "geometryType", "recordCount", "field_name", "alias", "type",
               "length", "nullable", "domainType", "domainName", "domainValues"]


def write_csv(report, path):
    rows = _field_rows(report)
    with open(path, "w", newline="", encoding="utf-8-sig") as fh:
        w = csv.DictWriter(fh, fieldnames=_FIELD_COLS)
        w.writeheader()
        w.writerows(rows)
    return path


def write_txt(report, path):
    L = []
    L.append("ArcGIS Online Feature Service Inventory")
    L.append("Generated: {}   Portal: {}".format(report["generated"], report.get("portal", "")))
    L.append("Services: {}".format(len(report["services"])))
    for svc in report["services"]:
        L.append("\n" + "=" * 72)
        L.append("{}  [{}]".format(svc["title"], svc["type"]))
        L.append("  id={}  owner={}  access={}".format(svc["id"], svc["owner"], svc["access"]))
        L.append("  created={}  modified={}".format(svc["created"], svc["modified"]))
        if svc["url"]:
            L.append("  url={}".format(svc["url"]))
        if svc["tags"]:
            L.append("  tags={}".format(svc["tags"]))
        for lyr in svc["layers"]:
            L.append("\n  -- {} '{}' (id {}) --".format(lyr["kind"], lyr["name"], lyr["id"]))
            L.append("     geometry={}  records={}  wkid={}  fields={}".format(
                lyr["geometryType"] or "n/a", lyr["recordCount"], lyr["wkid"], lyr["fieldCount"]))
            for f in lyr["fields"]:
                line = "       - {} ({}) {}".format(
                    f["name"], f["type"], "[{}]".format(f["alias"]) if f["alias"] else "")
                meta = []
                if f["length"]:
                    meta.append("len={}".format(f["length"]))
                meta.append("nullable={}".format(f["nullable"]))
                if f["domainType"]:
                    meta.append("domain={}:{}".format(f["domainType"], f["domainName"]))
                line += "  " + ", ".join(meta)
                L.append(line)
                if f["domainValues"]:
                    L.append("           values: {}".format(f["domainValues"]))
    Path(path).write_text("\n".join(L) + "\n", encoding="utf-8")
    return path


def write_xlsx(report, path):
    from openpyxl import Workbook
    from openpyxl.styles import Font

    wb = Workbook()
    bold = Font(bold=True)

    ws = wb.active
    ws.title = "Services"
    svc_cols = ["title", "id", "owner", "type", "access", "created", "modified",
                "layerCount", "url", "tags"]
    ws.append(svc_cols)
    for c in ws[1]:
        c.font = bold
    for svc in report["services"]:
        ws.append([svc["title"], svc["id"], svc["owner"], svc["type"], svc["access"],
                   svc["created"], svc["modified"], len(svc["layers"]), svc["url"], svc["tags"]])

    wl = wb.create_sheet("Layers")
    lyr_cols = ["service_title", "service_id", "layer_name", "kind",
                "geometryType", "recordCount", "wkid", "fieldCount"]
    wl.append(lyr_cols)
    for c in wl[1]:
        c.font = bold
    for svc in report["services"]:
        for lyr in svc["layers"]:
            wl.append([svc["title"], svc["id"], lyr["name"], lyr["kind"],
                       lyr["geometryType"], lyr["recordCount"], lyr["wkid"], lyr["fieldCount"]])

    wf = wb.create_sheet("Fields")
    wf.append(_FIELD_COLS)
    for c in wf[1]:
        c.font = bold
    for row in _field_rows(report):
        wf.append([row[k] for k in _FIELD_COLS])

    for sheet in wb.worksheets:
        for col in sheet.columns:
            width = min(60, max(10, max((len(str(c.value)) for c in col if c.value is not None), default=10) + 2))
            sheet.column_dimensions[col[0].column_letter].width = width

    wb.save(path)
    return path


def write_report(report, out_path, fmt):
    """fmt in {'TXT','CSV','XLSX'}; returns the written path."""
    fmt = (fmt or "TXT").upper()
    if fmt == "CSV":
        return write_csv(report, out_path)
    if fmt == "XLSX":
        return write_xlsx(report, out_path)
    return write_txt(report, out_path)


def build_report(gis, services, count_records=True, say=_noop):
    """Describe a list of items -> the report dict the writers consume."""
    return {
        "generated": datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%SZ"),
        "portal": _g(getattr(gis, "properties", None), "portalName", ""),
        "services": [describe_item(it, count_records, say) for it in services if it],
    }

## 5 - Run

In [ ]:
# ---- connect (no credentials in the notebook) ------------------------------
from arcgis.gis import GIS
try:
    gis = GIS("home")                          # ArcGIS Notebooks / hosted
except Exception:
    gis = GIS("pro")                           # running inside ArcGIS Pro
print("Signed in as", gis.users.me.username, "(", _g(getattr(gis, "properties", None), "portalName", ""), ")")

# ---- gather the target services for the chosen scope -----------------------
services = gather_services(
    gis, SCOPE, folders=FOLDERS or None, item_id_or_url=ID_OR_URL or None,
    search_query=SEARCH_TEXT or None, max_items=MAX_ITEMS)
print("{} feature service(s) to describe.".format(len(services)))

# ---- describe + write the log ----------------------------------------------
if not services:
    raise SystemExit("No feature services matched the chosen scope.")

report = build_report(gis, services, count_records=COUNT_RECORDS, say=print)

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
ext = {"XLSX": ".xlsx", "CSV": ".csv", "TXT": ".txt"}[OUTPUT_FORMAT.upper()]
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
base = OUTPUT_NAME or "agol_inventory_{}".format(stamp)
out_path = os.path.join(OUTPUT_DIR, base + ext)
write_report(report, out_path, OUTPUT_FORMAT)
print("\nLog written:", out_path)